# NOTEBOOK 06 – FINAL | TỔNG HỢP TOÀN BỘ ĐỒ ÁN NHÓM 02

**Đề tài chính thức:** Phân tích dữ liệu đơn hàng và dự đoán nguy cơ giao hàng trễ trong thương mại điện tử.

**Phụ trách tích hợp:** Tấn Lên (nhóm trưởng), **toàn bộ 5 thành viên cùng rà soát**.

**CHỈ CHỐT SAU KHI** 5 notebook cá nhân và các báo cáo Ch1–Ch6 đã hoàn thành, đóng băng split/model/threshold và bàn giao kết quả thật.

**Đầu vào dự kiến:** file data Ch3, dự đoán Test Ch4, chỉ số/hình Ch5, mô hình Ch4 và dashboard Ch6. Khi chưa có, notebook sẽ thông báo chưa đủ đầu vào; không giả lập số liệu.

[Xem hướng dẫn tích hợp](../docs/theory/06_final_integration.md) · [Checklist bàn giao](../docs/HANDOFF_CHECKLIST.md).

## 0. Mục lục & thứ tự chạy
1. Kiểm tra nguồn và tính tái lập.
2. Chuẩn hóa câu hỏi nghiên cứu, thời điểm t0, nhãn và đơn vị mẫu.
3. Kiểm tra chất lượng dữ liệu/EDA do Lam Linh bàn giao.
4. Kiểm tra giao thức Train/Validation/Test và so sánh Dummy–RF–MLP do Trúc Chi bàn giao.
5. Kiểm tra bảng metric Test/PR Curve/Top-K do Vĩnh Chi tính.
6. Tóm tắt insight và nêu câu trả lời cho mục tiêu/giả thuyết.
7. Kiểm tra Dashboard & file report, chốt giới hạn và phụ lục.

**Không dùng notebook final để train lại trên Test hoặc thay đổi ngưỡng sau khi có Test.**

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 1. Kiểm kê file bàn giao – điều kiện để chạy báo cáo cuối

In [ ]:
from IPython.display import display
needed = {
    "Ch3: dữ liệu cấp đơn": PROCESSED / "orders_model_ready.csv",
    "Ch4: frozen Test prediction": PROCESSED / "test_predictions.csv",
    "Ch4: metadata/threshold": PROCESSED / "experiment_metadata.json",
    "Ch4: champion model": ROOT / "models" / "champion.joblib",
    "Ch5: bảng metrics": ROOT / "reports" / "tables" / "model_metrics.csv",
    "Ch5: Top-K": ROOT / "reports" / "tables" / "topk_analysis.csv",
    "Ch5: hình PR Curve": ROOT / "reports" / "figures" / "pr_curve_test.png",
    "Ch6: Dashboard": ROOT / "app" / "app.py",
}
manifest = pd.DataFrame([
    {"sản phẩm":name, "tệp":str(p.relative_to(ROOT)), "sẵn có":p.exists()}
    for name,p in needed.items()
])
display(manifest)
if not manifest["sẵn có"].all():
    print("BẢN NHÁP: chưa đầy đủ đầu ra. Cần bàn giao các tệp thiếu trước khi bảo vệ.")
    raise FileNotFoundError("Thiếu artifact, không được ghi kết luận final giả.")

## 2. Kiểm tra dữ liệu, nhãn và thời điểm dự báo
**Bài toán:** dự báo tại `order_approved_at`; `is_late` dựa trên ngày giao thực tế > ngày giao dự kiến.

Đầu ra kiểm tra: số đơn có nhãn, số đơn trễ, tỷ lệ trễ, kiểm tra uniqueness; nhớ ghi phạm vi delivered-only và nguồn Brazil 2016–2018.

In [ ]:
from src.contracts import assert_unique_orders
import json, joblib
data = pd.read_csv(PROCESSED / "orders_model_ready.csv")
assert_unique_orders(data)
assert data["is_late"].isin([0, 1]).all()
late_count = int((data["is_late"] == 1).sum())
display(pd.DataFrame([{
    "orders_with_labels":len(data),
    "late_orders":late_count,
    "on_time_orders":int(len(data)-late_count),
    "late_rate":late_count/len(data) if len(data) else float("nan")
}]))
print("Thống kê là của tập delivered có nhãn (nếu đã chạy Ch3).")

## 3. Kiểm chứng mô hình và tính nhất quán Test

Nhãn Test dự đoán từ **duy nhất một phiên bản mô hình**, ngưỡng từ Validation. Không báo một bộ số trong notebook và một bộ khác trong báo cáo.

In [ ]:
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
pred = pd.read_csv(PROCESSED / "test_predictions.csv")
metric_table = pd.read_csv(ROOT / "reports/tables/model_metrics.csv")
meta = json.loads((PROCESSED / "experiment_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ROOT / "models/champion.joblib")  # chỉ dùng artifact nội bộ đã tin cậy
assert pred["order_id"].is_unique
assert pred["y_true"].isin([0,1]).all()
assert meta["champion"] in {"rf","mlp"}
champion = meta["champion"]
threshold = float(meta["chosen_threshold"])
check_pred = (pred["p_"+champion] >= threshold).astype(int)
assert np.array_equal(check_pred, pred["y_pred_champion"].to_numpy()), "Sai khác ngưỡng/y_pred."
review = metric_table[metric_table["model"].eq(champion)]
assert len(review) == 1, "Không có duy nhất 1 dòng champion trong model_metrics."
check_AP = average_precision_score(pred["y_true"], pred["p_"+champion])
assert np.isclose(check_AP, review["AP"].iloc[0], rtol=1e-6), "AP không khớp notebook Ch5."
display(metric_table.sort_values("AP",ascending=False))
print("Đã đối chiếu AP và threshold trên frozen Test.")

## 4. Tổng hợp insight EDA và Top-K có chứng cứ

Giải thích tỷ lệ giao trễ lịch sử từ dữ liệu, và ý nghĩa ứng dụng ưu tiên K đơn. Không suy ra quan hệ nhân quả hoặc hiệu quả triển khai thời gian thực khi chỉ có dữ liệu hồi cứu.

In [ ]:
topk = pd.read_csv(ROOT / "reports" / "tables" / "topk_analysis.csv")
display(topk)
by_state = data.groupby("customer_state", dropna=False, as_index=False).agg(
    orders=("order_id","nunique"), late_rate=("is_late","mean")
).sort_values("orders",ascending=False)
display(by_state.head(15))
print("Hãy bổ sung 3–5 insight từ số liệu thật và nêu mẫu số của mỗi tỷ lệ.")

## 5. Bộ bằng chứng báo cáo và Dashboard
Yêu cầu kiểm tra:
- [ ] 6 chương + phụ lục đã thống nhất với `docs/REPORT_OUTLINE.md`.
- [ ] Có hình PR Curve `reports/figures/pr_curve_test.png`.
- [ ] KPI/metrics trùng notebook Ch5 và tập Test duy nhất.
- [ ] Dashboard Streamlit chạy offline và mô tả là demo hồi cứu.
- [ ] README, requirements, nguồn Kaggle và hướng dẫn tái lập được kiểm tra.
- [ ] Cả 5 thành viên xác nhận kiểm tra mô tả chức năng/phương pháp thuộc mình.

In [ ]:
chapters = sorted((ROOT / "reports" / "chapters").glob("*.md"))
display(pd.DataFrame([
    {"report_section":p.name, "size_bytes":p.stat().st_size}
    for p in chapters
]))
print("Lưu ý: file tồn tại không đồng nghĩa nội dung đã hoàn chỉnh; phải review thủ công.")
from IPython.display import Image
display(Image(filename=str(ROOT / "reports" / "figures" / "pr_curve_test.png")))

## 6. Bố cục nội dung thuyết trình final

1. **Vấn đề:** phát hiện nguy cơ giao trễ sớm, giá trị với vận hành.
2. **Dataset & nhãn:** Olist, các bảng và lý do tạo 1 dòng/order_id.
3. **EDA:** thống kê và insight có minh chứng.
4. **Mô hình:** Dummy / RF / MLP, temporal split, tránh Data Leakage.
5. **Kết quả Test:** bảng Precision/Recall/F1/AP, PR curve, cảnh báo sai và Top-K.
6. **Dashboard:** demo lịch sử và danh sách đơn ưu tiên.
7. **Hạn chế:** delivered-only, dữ liệu 2016–2018 Brazil và không có vận hành realtime.
8. **Kết luận:** trả lời đúng mục tiêu ban đầu.

**Nguyên tắc:** không thêm bảng số liệu, hình biểu đồ hoặc kết luận mới nếu chưa kiểm tra khả năng tái lập.

## 7. Ký nhận bàn giao & điều kiện phát hành
- [ ] **Tấn Lên:** Ch1, Ch6, phụ lục và chạy dashboard; nghiệm thu final.
- [ ] **Ngọc Lan:** Ch2 khớp thuật toán/độ đo thực sự dùng, nguồn trích dẫn đủ.
- [ ] **Lam Linh:** Ch3 dataset/schema/EDA tái lập được, audit leakage.
- [ ] **Trúc Chi:** Ch4 split/model/threshold và Test predictions khớp.
- [ ] **Vĩnh Chi:** Ch5 bảng metrics/PR/top-K và hạn chế đúng số liệu.

**Hạn nội bộ:** bản đầy đủ 27/11, phản biện 28–29/11, phát hành 30/11/2026.

**Trạng thái file lúc khởi tạo:** KHUNG FINAL – CHƯA CÓ KẾT QUẢ OLIST.